[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/01.%20Parte%201/06.%20Clase%206/06Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=01.%20Parte%201%2F06.%20Clase%206%2F06Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 6: Media robusta (estimador de Huber) y generación de números aleatorios

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

---

## Introducción

En la Clase 5 mejoramos la estimación de la covarianza con **Shrunk Covariance**. En esta clase abordamos el otro insumo crítico de la optimización de portafolios: el **vector de rendimientos esperados** $\boldsymbol{\mu}$.

La media muestral $\bar{r} = \frac{1}{T}\sum_{t=1}^{T} r_t$ es muy sensible a **valores extremos** (outliers). Un solo día con un rendimiento atípico puede sesgar significativamente la estimación.

### Objetivos
1. Introducir el **estimador de Huber** como alternativa robusta a la media muestral.
2. Comparar media muestral vs. Huber en datos financieros reales.
3. Revisar los fundamentos de **generación de números aleatorios** (LCG, Box-Muller) como base para simulación Monte Carlo.
4. Medir el impacto de estimadores robustos en portafolios.

## 1. Paquetes y datos

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from sklearn.covariance import LedoitWolf

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 10)

In [ ]:
tickers = ["AA", "AAPL", "AMZN", "MSFT", "KO", "NVDA"]
start_date = "2025-01-01"
end_date   = "2025-03-27"

data = yf.download(tickers, start=start_date, end=end_date,
                   auto_adjust=True, progress=False)
closes = data["Close"].dropna()
daily_returns = np.log(closes / closes.shift(1)).dropna()
daily_returns.describe()

---
## 2. Estimador de Huber para la media

### El problema de los outliers

La media muestral minimiza la suma de errores al cuadrado:

$$
\bar{r} = \arg\min_\mu \sum_{t=1}^{T} (r_t - \mu)^2
$$

Esta función de pérdida penaliza **cuadráticamente** las observaciones extremas, dándoles una influencia desproporcionada.

### Función de pérdida de Huber

El estimador de Huber (Huber, 1964) usa una función de pérdida **híbrida**: cuadrática para residuos pequeños y lineal para residuos grandes:

$$
\rho_c(x) = \begin{cases} \frac{1}{2}x^2 & \text{si } |x| \leq c \\ c|x| - \frac{1}{2}c^2 & \text{si } |x| > c \end{cases}
$$

donde $c > 0$ es el **parámetro de corte** (*tuning constant*). Valores típicos: $c = 1.345$ (eficiencia del 95% bajo normalidad).

### Interpretación

- Observaciones con $|r_t - \mu| \leq c \cdot \sigma$: se tratan como en la media clásica.
- Observaciones con $|r_t - \mu| > c \cdot \sigma$: su influencia se **limita** linealmente.

Resultado: el estimador es **resistente a outliers** sin ignorarlos completamente.

In [ ]:
# Visualizar la función de pérdida de Huber vs. cuadrática
x = np.linspace(-4, 4, 500)
c = 1.345

loss_quadratic = 0.5 * x**2
loss_huber = np.where(np.abs(x) <= c, 0.5 * x**2, c * np.abs(x) - 0.5 * c**2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, loss_quadratic, label="Cuadrática (media muestral)", linewidth=2)
ax.plot(x, loss_huber, label=f"Huber (c = {c})", linewidth=2, linestyle="--")
ax.axvline(-c, color="gray", linestyle=":", alpha=0.5)
ax.axvline(c, color="gray", linestyle=":", alpha=0.5, label=f"±c = ±{c}")
ax.set_xlabel("Residuo (x)")
ax.set_ylabel("Pérdida ρ(x)")
ax.set_title("Función de pérdida: cuadrática vs. Huber")
ax.legend()
plt.tight_layout()

---
## 3. Comparación: media muestral vs. Huber

### Implementación con statsmodels

`statsmodels.robust.scale.Huber()` estima simultáneamente la **media** y la **escala** (desviación estándar) de forma robusta.

In [ ]:
huber = sm.robust.scale.Huber()

results = []
for col in daily_returns.columns:
    data_col = daily_returns[col].values
    mu_sample = data_col.mean()
    mu_huber, scale_huber = huber(data_col)
    results.append({
        "Activo": col,
        "Media muestral": mu_sample,
        "Media Huber": mu_huber,
        "Diferencia (%)": (mu_huber - mu_sample) / abs(mu_sample) * 100 if mu_sample != 0 else 0,
        "Escala Huber": scale_huber,
        "Std muestral": data_col.std()
    })

comparison = pd.DataFrame(results).set_index("Activo")
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Media
comp_mu = comparison[["Media muestral", "Media Huber"]]
comp_mu.plot(kind="bar", ax=axes[0])
axes[0].set_title("Media diaria: muestral vs. Huber")
axes[0].set_ylabel("Rendimiento medio diario")
axes[0].tick_params(axis='x', rotation=0)

# Escala
comp_sigma = comparison[["Std muestral", "Escala Huber"]]
comp_sigma.plot(kind="bar", ax=axes[1])
axes[1].set_title("Dispersión: std muestral vs. escala Huber")
axes[1].set_ylabel("Desviación / Escala")
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle("Estimadores clásicos vs. robustos", fontsize=14)
plt.tight_layout()

### Efecto de outliers artificiales

Para ilustrar la robustez, contaminamos los datos con un outlier extremo y comparamos ambos estimadores:

In [ ]:
# Contaminar AAPL con un outlier extremo
contaminated = daily_returns["AAPL"].copy()
contaminated.iloc[5] = -0.30  # Crash artificial del 30%

mu_sample_clean = daily_returns["AAPL"].mean()
mu_sample_dirty = contaminated.mean()
mu_huber_clean, _ = huber(daily_returns["AAPL"].values)
mu_huber_dirty, _ = huber(contaminated.values)

print("AAPL — Efecto de un outlier (-30%):")
print(f"  Media muestral limpia:       {mu_sample_clean:.6f}")
print(f"  Media muestral contaminada:  {mu_sample_dirty:.6f}  (cambio: {(mu_sample_dirty-mu_sample_clean)/abs(mu_sample_clean)*100:+.1f}%)")
print(f"  Media Huber limpia:          {mu_huber_clean:.6f}")
print(f"  Media Huber contaminada:     {mu_huber_dirty:.6f}  (cambio: {(mu_huber_dirty-mu_huber_clean)/abs(mu_huber_clean)*100:+.1f}%)")

---
## 4. Generación de números aleatorios

La simulación Monte Carlo requiere generar secuencias de números **pseudoaleatorios**. Revisamos dos métodos fundamentales.

### 4.1 Generador congruencial lineal (LCG)

El **LCG** genera una secuencia de enteros mediante la recurrencia:

$$
x_{n+1} = (a \cdot x_n + c) \mod m
$$

donde:
- $m$: módulo (determina el período máximo)
- $a$: multiplicador
- $c$: incremento
- $x_0$: semilla (*seed*)

Los números uniformes en $[0, 1)$ se obtienen como $u_n = x_n / m$.

**Generador mínimo estándar** (Park & Miller, 1988): $m = 2^{31} - 1$, $a = 16807$, $c = 0$.

In [ ]:
def lcg(n, m=2**31-1, a=16807, c=0, seed=2**30):
    """Generador congruencial lineal."""
    x = np.zeros(n + 1)
    x[0] = seed
    for i in range(1, n + 1):
        x[i] = (a * x[i-1] + c) % m
    return x[1:] / m

# Ejemplo pequeño
print("LCG con m=31, a=13, c=0, seed=3:")
print(lcg(10, m=31, a=13, c=0, seed=3))

In [ ]:
# Generador mínimo estándar: 10,000 uniformes
x_uniform = lcg(10000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(x_uniform, bins=50, stat="density", ax=axes[0], alpha=0.6)
axes[0].axhline(1.0, color="red", linestyle="--", label="U(0,1) teórica")
axes[0].set_title("LCG — Generador mínimo estándar")
axes[0].set_xlabel("u")
axes[0].legend()

# RANDU (IBM) — ejemplo de mal generador
x_randu = lcg(10000, m=2**31, a=2**16+3, c=0, seed=3)
sns.histplot(x_randu, bins=50, stat="density", ax=axes[1], alpha=0.6, color="orange")
axes[1].axhline(1.0, color="red", linestyle="--", label="U(0,1) teórica")
axes[1].set_title("RANDU (IBM) — generador defectuoso")
axes[1].set_xlabel("u")
axes[1].legend()

plt.suptitle("Distribución de generadores congruenciales", fontsize=14)
plt.tight_layout()

### 4.2 Método de Box-Muller

El método de **Box-Muller** (1958) transforma dos uniformes independientes $U_1, U_2 \sim U(0,1)$ en dos normales estándar independientes:

$$
Z_1 = \sqrt{-2\ln(1 - U_1)} \cos(2\pi U_2)
$$

$$
Z_2 = \sqrt{-2\ln(1 - U_1)} \sin(2\pi U_2)
$$

Esto permite generar rendimientos normales $r \sim \mathcal{N}(\mu, \sigma^2)$ como $r = \mu + \sigma Z$.

In [ ]:
def box_muller(n, seed=2**30):
    """Genera n normales estándar via Box-Muller usando LCG."""
    u = lcg(n, seed=seed)
    u1 = u[:n//2]
    u2 = u[n//2:2*(n//2)]
    z1 = np.sqrt(-2 * np.log(1 - u1)) * np.cos(2 * np.pi * u2)
    z2 = np.sqrt(-2 * np.log(1 - u1)) * np.sin(2 * np.pi * u2)
    return np.concatenate([z1, z2])

z = box_muller(100000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Histogram
sns.histplot(z, bins=80, stat="density", ax=axes[0], alpha=0.6)
x_norm = np.linspace(-4, 4, 200)
axes[0].plot(x_norm, stats.norm.pdf(x_norm), "r--", linewidth=2, label="N(0,1) teórica")
axes[0].set_title("Box-Muller: histograma vs. normal teórica")
axes[0].legend()

# QQ-plot
stats.probplot(z[:5000], dist="norm", plot=axes[1])
axes[1].set_title("Box-Muller: QQ-plot")

plt.suptitle("Verificación del método de Box-Muller", fontsize=14)
plt.tight_layout()

---
## 5. Aplicación: simulación Monte Carlo de rendimientos

Combinamos los conceptos anteriores para simular trayectorias de precios y rendimientos de un portafolio, usando estimadores robustos (Huber + Ledoit-Wolf).

In [ ]:
# Estimadores robustos
mu_huber_vec = np.array([huber(daily_returns[col].values)[0] for col in daily_returns.columns])
lw = LedoitWolf().fit(daily_returns)
Sigma_lw = lw.covariance_

print("Rendimientos esperados (Huber):")
for ticker, mu in zip(daily_returns.columns, mu_huber_vec):
    print(f"  {ticker}: {mu:.6f}")
print(f"\nα Ledoit-Wolf: {lw.shrinkage_:.4f}")

In [ ]:
# Simulación Monte Carlo: 10,000 portafolios aleatorios
np.random.seed(42)
n_assets = len(daily_returns.columns)
n_portfolios = 10000

weights = np.random.dirichlet(np.ones(n_assets), n_portfolios)
port_returns = 252 * weights @ mu_huber_vec
port_risks = np.array([np.sqrt(252 * w @ Sigma_lw @ w) for w in weights])
sharpe = port_returns / port_risks

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(port_risks, port_returns, c=sharpe, cmap="RdYlBu",
                     s=5, alpha=0.5)
plt.colorbar(scatter, label="Ratio de Sharpe")
ax.set_xlabel("Riesgo (σ anualizada)")
ax.set_ylabel("Rendimiento esperado (anualizado)")
ax.set_title("Simulación Monte Carlo: 10,000 portafolios\n(μ Huber + Σ Ledoit-Wolf)")

# Marcar máximo Sharpe
idx_best = sharpe.argmax()
ax.scatter(port_risks[idx_best], port_returns[idx_best], marker="*",
           s=400, color="red", zorder=5, label=f"Max Sharpe = {sharpe[idx_best]:.3f}")
ax.legend()
plt.tight_layout()

---

## 6. Referencias bibliográficas

- **Box, G. E. P. & Muller, M. E.** (1958). A Note on the Generation of Random Normal Deviates. *The Annals of Mathematical Statistics*, 29(2), 610–611.
- **Huber, P. J.** (1964). Robust Estimation of a Location Parameter. *The Annals of Mathematical Statistics*, 35(1), 73–101.
- **Huber, P. J. & Ronchetti, E. M.** (2009). *Robust Statistics* (2nd ed.). Wiley.
- **Ledoit, O. & Wolf, M.** (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, 88(2), 365–411.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press. — Cap. 6–8.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **Michaud, R. O.** (1989). The Markowitz Optimization Enigma. *Financial Analysts Journal*, 45(1), 31–42.
- **Park, S. K. & Miller, K. W.** (1988). Random Number Generators: Good Ones Are Hard To Find. *Communications of the ACM*, 31(10), 1192–1201.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning.